In [ ]:
## Tokenizer comes before the actual model - BERT Training. 
## Training it means we are just learning the vocabulary of the model and the rules for breaking raw text into tokens.
## Essentially statistical learning of the language.

### Imports

In [2]:
from pathlib import Path
import json
import gzip            
print("imports complete")

imports complete


### Define Paths

In [5]:
current_directory = Path.cwd()              # gives the current working directory
data_directory = current_directory / "data"       # creates a path to the data directory
train_path = data_directory / "enwiki_train.jsonl"  
validation_path = data_directory / "enwiki_validation.jsonl"  

print("Current directory :", current_directory)
print("Train path        :", train_path)
print("Validation path   :", validation_path)

print()
print("Train exists      :", train_path.exists())
print("Validation exists :", validation_path.exists())

Current directory : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model
Train path        : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data\enwiki_train.jsonl
Validation path   : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data\enwiki_validation.jsonl

Train exists      : True
Validation exists : True


### Text Iterator

In [6]:
## a generator that streams only the text field from the training JSONL
## rather than loading the entire file into memory at once. This is important for large datasets.

In [7]:
def training_text_iterator(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:                                  # reads the file one JSONL record at a time
            record = json.loads(line)                      # convert JSON to python dictionary
            yield record['text']                           # if we used return instead of yield, 
                                                           # it would return the first record and exit the function. 
                                                           # Yield allows us to return one record at a time 
                                                           # and continue from where we left off.

In [10]:
## verification
text_iterator = training_text_iterator(train_path)        # creates a generator object for the training file. Does not read all documents yet

for i in range(3):

    text = next(text_iterator)                            # asks the generator for it's next document
                                                          # next(...) is what causes the generator to resume execution until it reaches the next yield
    
    print('-' * 30)
    print(f'Document {i+1}')
    print('-' * 30)

    print(text[:500])
    print()

------------------------------
Document 1
------------------------------
is a retired two-star general of the Pakistan Army, who served as the Director General Infantry at the General Headquarters in Rawalpindi.

Education
Khan has done his PhD in Peace and Conflict Studies.

Military career
Khan was commissioned into the 34 Punjab Regiment via the 22 OTS Course. In 2018, Khan was promoted to the rank Major General from Brigadier. As Force Commander Northern Areas, he oversaw military operations and civil-military initiatives in Gilgit-Baltistan. Later he served as D

------------------------------
Document 2
------------------------------
(19 June 1896 – 9 November 1975) was a Scottish-born medical practitioner, philanthropist and humanitarian aid worker. She was for many years a general practitioner (GP) in Harrogate, West Yorkshire, England. She was known locally as and .

Rutherford trained as a physician in Glasgow and Leeds, gained experience as a senior house surgeon in the Nort

### Import Tokenizer components

In [15]:
import tokenizers                          # HF's main tokenizers library
from tokenizers import Tokenizer           # imports the main tokenizer container/object that we will configure
from tokenizers.models import  WordPiece   # Imports WP tokenization model (subword algo used by BERT)
from tokenizers.trainers import WordPieceTrainer    # the component that learns the WP vocabulary from our corpus
from tokenizers.normalizers import NFC   # Imports BERT-style text normalization, such as handling whitespace, accents, and case depending on configuration.
from tokenizers.pre_tokenizers import BertPreTokenizer  # imports BERT's initial text-splitting logic before WordPiece is applied

print(tokenizers.__version__)

0.22.2


### Tokenizer Architecture

In [ ]:
TOKENIZER CONFIGURATION

Tokenization level              : Subword
Algorithm                       : WordPiece
Vocabulary size                 : 21,000
Minimum frequency               : 5

Casing                          : Cased
Unicode normalization           : NFC
Accent stripping                : None
Pre-tokenization                : BERT-style

Continuing subword prefix       : ##
Initial alphabet limit          : 1,000
Max input characters per word   : 100

Special tokens:
[PAD]
[UNK]
[CLS]
[SEP]
[MASK]

In [13]:
wordpiece_model = WordPiece(                 # creates the model, does not the have vocabulary at this stage 
    unk_token="[UNK]",                       # handles unknown inputs
    continuing_subword_prefix="##",          # marks tokens with ## that are continuation of a word
    max_input_chars_per_word=100             # the input string cannot be > 100 characters
)

tokenizer = Tokenizer(wordpiece_model)       # creates the main tokenizer object

print("WP tokenzier created.")

WP tokenzier created.


In [16]:
## configure NFC - Normalization Form Canonical Coversion - converts characters into a canonical pre-composed form
tokenizer.normalizer = NFC()
print("NFC normalization confirmed")

NFC normalization confirmed


In [17]:
## configure BERT style pre-tokenizer (separates strings using whitespaces and punctuations)
tokenizer.pre_tokenizer = BertPreTokenizer()
print("BERT-style pre-tokenization configured.")

BERT-style pre-tokenization configured.


In [18]:
## define special tokens
special_tokens = [
    "[PAD]",                # to pad the length of each input to a standard tensor shape
    "[UNK]",                # handles unknown input
    "[CLS]",                # classifier for downstream classification tasks
    "[SEP]",                # used as a separator token in between sentences
    "[MASK]"                # will be used to mask words during training
]
print("Special tokens:", special_tokens)

Special tokens: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']


In [34]:
## WordPiece Trainer -  basically describes vocabulary learning and later the model initialized above will be used for tokenization or actually encoding input
trainer = WordPieceTrainer(
    vocab_size=31000,
    min_frequency=5,
    limit_alphabet=1000,
    continuing_subword_prefix="##",
    special_tokens=special_tokens
)

print("WordPiece trainer created.")

WordPiece trainer created.


In [35]:
## create training iterator
train_iterator = training_text_iterator(train_path)
print("Training iterator created.")

Training iterator created.


### Train Tokenizer on full corpus

In [36]:
tokenizer.train_from_iterator(
    train_iterator,
    trainer=trainer
)

print("Tokenizer training complete")
print("Vocabular size :", tokenizer.get_vocab_size())

Tokenizer training complete
Vocabular size : 31000


In [37]:
## Inspect vocab entries
vocab = tokenizer.get_vocab()               # key = token, value = token id

vocab_by_id = sorted(
    vocab.items(),
    key = lambda item: item[1]
)

print("Vocab size :", len(vocab_by_id))

for token, token_id in vocab_by_id[:100]:
    print(f"{token_id:5d}  {token}")

Vocab size : 31000
    0  [PAD]
    1  [UNK]
    2  [CLS]
    3  [SEP]
    4  [MASK]
    5  !
    6  "
    7  #
    8  $
    9  %
   10  &
   11  '
   12  (
   13  )
   14  *
   15  +
   16  ,
   17  -
   18  .
   19  /
   20  0
   21  1
   22  2
   23  3
   24  4
   25  5
   26  6
   27  7
   28  8
   29  9
   30  :
   31  ;
   32  <
   33  =
   34  >
   35  ?
   36  @
   37  A
   38  B
   39  C
   40  D
   41  E
   42  F
   43  G
   44  H
   45  I
   46  J
   47  K
   48  L
   49  M
   50  N
   51  O
   52  P
   53  Q
   54  R
   55  S
   56  T
   57  U
   58  V
   59  W
   60  X
   61  Y
   62  Z
   63  [
   64  \
   65  ]
   66  ^
   67  _
   68  `
   69  a
   70  b
   71  c
   72  d
   73  e
   74  f
   75  g
   76  h
   77  i
   78  j
   79  k
   80  l
   81  m
   82  n
   83  o
   84  p
   85  q
   86  r
   87  s
   88  t
   89  u
   90  v
   91  w
   92  x
   93  y
   94  z
   95  {
   96  |
   97  }
   98  ~
   99  ¢


In [38]:
for start in [1000, 5000, 10000, 15000, 20000]:
    
    print("\n" + "=" * 30)
    print(f"Vocabulary IDs {start}–{start + 19}")
    print("=" * 30)

    for token, token_id in vocab_by_id[start:start + 20]:
        print(f"{token_id:5d}  {token}")


Vocabulary IDs 1000–1019
 1000  （
 1001  ）
 1002  ，
 1003  ：
 1004  ｜
 1005  ##r
 1006  ##i
 1007  ##d
 1008  ##w
 1009  ##e
 1010  ##o
 1011  ##b
 1012  ##h
 1013  ##ü
 1014  ##n
 1015  ##a
 1016  ##u
 1017  ##s
 1018  ##k
 1019  ##x

Vocabulary IDs 5000–5019
 5000  Portug
 5001  Bibl
 5002  adop
 5003  deep
 5004  ##ologist
 5005  response
 5006  Ros
 5007  outside
 5008  Gre
 5009  lived
 5010  ##ands
 5011  organiz
 5012  radio
 5013  east
 5014  pe
 5015  ##ym
 5016  aim
 5017  feature
 5018  Service
 5019  1985

Vocabulary IDs 10000–10019
10000  ##sch
10001  ##aba
10002  Brooklyn
10003  Lithuanian
10004  09
10005  1894
10006  SE
10007  ##hing
10008  merged
10009  Howard
10010  roads
10011  ##arks
10012  EF
10013  tomb
10014  ##door
10015  hearing
10016  ##active
10017  Operation
10018  ##esters
10019  ##itals

Vocabulary IDs 15000–15019
15000  Zurich
15001  Sag
15002  ##vor
15003  ##overed
15004  legally
15005  Acting
15006  climbing
15007  Nancy
15008  Wak
15009  ##orig
15010  

In [39]:
## test the tokenization process
test_sentences = [
    "The patient has patellofemoral chondromalacia.",
    "The player was playing football.",
    "Neurodegenerative diseases affect the nervous system.",
    "COVID-19 caused a global pandemic.",
]

for text in test_sentences:

    encoding = tokenizer.encode(text)

    print('\n Text :')
    print(text)
    print()
    print("tokens :")
    print(encoding.tokens)


 Text :
The patient has patellofemoral chondromalacia.

tokens :
['The', 'patient', 'has', 'pat', '##ello', '##fe', '##mor', '##al', 'cho', '##nd', '##rom', '##ala', '##cia', '.']

 Text :
The player was playing football.

tokens :
['The', 'player', 'was', 'playing', 'football', '.']

 Text :
Neurodegenerative diseases affect the nervous system.

tokens :
['Neur', '##ode', '##gen', '##era', '##ti', '##ve', 'diseases', 'affect', 'the', 'nervous', 'system', '.']

 Text :
COVID-19 caused a global pandemic.

tokens :
['COVID', '-', '19', 'caused', 'a', 'global', 'pandemic', '.']


In [40]:
## ^^The tokenizer apparently did not allocate complete tokens for these medical terms. 
## But crucially, it can still construct them entirely from learned subwords.

### Tokenizer Stress Test

In [41]:
test_groups = {
    
    "General English": [
        "The patient went to the hospital.",
        "Machine learning is transforming the world.",
        "The government announced a new policy."
    ],
    
    "Medical": [
        "The patient has patellofemoral chondromalacia.",
        "Neurodegenerative disorders affect the nervous system.",
        "The MRI demonstrated osteoarthritis and meniscal degeneration."
    ],
    
    "Biomedical": [
        "BRCA1 mutations are associated with breast cancer.",
        "TNF-alpha regulates inflammatory responses.",
        "Interleukin-6 plays an important role in inflammation."
    ],
    
    "Scientific notation": [
        "The dosage was 5 mg/dL.",
        "Blood pressure was 120 mmHg.",
        "The lesion measured 3.5-mm."
    ],
    
    "Multilingual / Unicode": [
        "The café serves excellent food.",
        "El paciente tiene dolor.",
        "रोगी को दर्द है।",
        "患者に痛みがあります。",
        "Пациент испытывает боль."
    ]
}


for group_name, sentences in test_groups.items():
    
    print("\n" + "=" * 30)
    print(group_name.upper())
    print("=" * 30)
    
    for text in sentences:
        
        encoding = tokenizer.encode(text)
        
        print("\nText   :", text)
        print("Tokens :", encoding.tokens)
        print("Count  :", len(encoding.tokens))


GENERAL ENGLISH

Text   : The patient went to the hospital.
Tokens : ['The', 'patient', 'went', 'to', 'the', 'hospital', '.']
Count  : 7

Text   : Machine learning is transforming the world.
Tokens : ['Machine', 'learning', 'is', 'transforming', 'the', 'world', '.']
Count  : 7

Text   : The government announced a new policy.
Tokens : ['The', 'government', 'announced', 'a', 'new', 'policy', '.']
Count  : 7

MEDICAL

Text   : The patient has patellofemoral chondromalacia.
Tokens : ['The', 'patient', 'has', 'pat', '##ello', '##fe', '##mor', '##al', 'cho', '##nd', '##rom', '##ala', '##cia', '.']
Count  : 14

Text   : Neurodegenerative disorders affect the nervous system.
Tokens : ['Neur', '##ode', '##gen', '##era', '##ti', '##ve', 'disorders', 'affect', 'the', 'nervous', 'system', '.']
Count  : 12

Text   : The MRI demonstrated osteoarthritis and meniscal degeneration.
Tokens : ['The', 'MR', '##I', 'demonstrated', 'ost', '##eo', '##arth', '##rit', '##is', 'and', 'men', '##iscal', 'deg', '

In [ ]:
## ^^The token did not encounter Hindi or Russian as languages. 
## It merely encountered enough of their characters while training on English Wikipedia
## for some of those characters to enter its 1000 character alphabet

## Eg - The english wikipedia might contain - Moscow (Russian: Москва) is the capital of Russia.
## because our limit_alphabet = 1000, these characters would make it into that

In [43]:
for group_name, sentences in test_groups.items():
    
    total_words = 0
    total_tokens = 0
    
    for text in sentences:
        
        encoding = tokenizer.encode(text)
        
        word_count = len(text.split())
        token_count = len(encoding.tokens)
        
        total_words += word_count
        total_tokens += token_count
    
    fragmentation_ratio = total_tokens / total_words
    
    print(
        f"{group_name:25s} "
        f"Words: {total_words:3d} | "
        f"Tokens: {total_tokens:3d} | "
        f"Tokens/Word: {fragmentation_ratio:.2f}"
    )

General English           Words:  18 | Tokens:  21 | Tokens/Word: 1.17
Medical                   Words:  18 | Tokens:  41 | Tokens/Word: 2.28
Biomedical                Words:  18 | Tokens:  37 | Tokens/Word: 2.06
Scientific notation       Words:  14 | Tokens:  29 | Tokens/Word: 2.07
Multilingual / Unicode    Words:  17 | Tokens:  47 | Tokens/Word: 2.76


In [44]:
for group_name, sentences in test_groups.items():
    
    unk_count = 0
    total_tokens = 0
    
    for text in sentences:
        
        encoding = tokenizer.encode(text)
        
        unk_count += encoding.tokens.count("[UNK]")
        total_tokens += len(encoding.tokens)
    
    print(
        f"{group_name:25s} "
        f"[UNK]: {unk_count:3d} | "
        f"Total tokens: {total_tokens:3d}"
    )

General English           [UNK]:   0 | Total tokens:  21
Medical                   [UNK]:   0 | Total tokens:  41
Biomedical                [UNK]:   0 | Total tokens:  37
Scientific notation       [UNK]:   0 | Total tokens:  29
Multilingual / Unicode    [UNK]:   3 | Total tokens:  47
